# Solar Filament Segmentation Challenge 2026 -- Pretraining Data Prep (Kaggle)

Clones the `jp-pretraining-data-prep` branch and drives
`scripts/pretrain_data/gong_halpha.py` (Stage 1: scrape + download GONG H-Alpha
FITS imagery) and `scripts/pretrain_data/preprocess_gong.py` (Stage 2: convert
FITS to JPEG, no other processing) -- see `PRETRAIN_PLAN.md` for the full
design and what's been verified against the real archive already.

This notebook does **not** need a GPU -- Stage 1/2 are CPU-only I/O and image
processing. Its job is to produce two Kaggle Datasets (`halpha-raw`,
`halpha-preprocessed`) that later pretraining notebooks mount as read-only
inputs, so this is the only notebook that needs internet access.

### 0. Clone the repo

Requires `jp-pretraining-data-prep` to already be pushed to `origin`.

In [ ]:
!git clone -b jp-pretraining-data-prep https://github.com/jprakash-1/Solar-Filament-Segmentation.git


In [ ]:
%cd Solar-Filament-Segmentation
!ls


In [ ]:
# re-run-safe: pick up any commits pushed after the kernel started
!git pull origin jp-pretraining-data-prep


### 1. Install dependencies

Only what Stage 1/2 actually need (`requests`, `beautifulsoup4`, `astropy`)
-- deliberately not the full `requirements.txt`, which also pulls in
`torch`/`torchvision`. Those aren't needed here, and blindly
`pip install`-ing them has separately bitten this project's `jp-mvp1` branch by
reinstalling a PyPI wheel that dropped kernel support for older Kaggle GPUs (see
that branch's README "Known gotchas") -- avoid the whole class of problem by not
installing them in a notebook that has no use for them.

In [ ]:
!pip install -q requests beautifulsoup4 astropy


### 2. Stage 1 -- build a manifest (no downloading yet)

Builds a `(site, timestamp, url)` manifest by walking the real GONG archive
(`https://gong2.nso.edu/HA/haf/...`, verified live -- see `PRETRAIN_PLAN.md`
section 2) and picking ~1 frame/hour per site. **Review the row count before
downloading** -- `--day-stride` controls how many days are sampled within each
date range, and needs tuning to land the final corpus in the 5,000-10,000 image
target range (2 sites' combined daylight coverage is ~15-20 frames/day, so
daily coverage across both multi-year windows would hugely overshoot that).

Start with a short date range first (a few days) to sanity-check the pipeline
end-to-end before committing to the full multi-year pull below -- rerunning
`manifest` is cheap (no downloads happen in this step).

In [ ]:
# Quick sanity check on a short, known-good range before the full pull
!python scripts/pretrain_data/gong_halpha.py manifest \
    --sites big_bear mauna_loa \
    --date-ranges 2022-03-18:2022-03-20 \
    --day-stride 1 \
    --workers 6 \
    --out /tmp/manifest_smoke_test.csv \
    --max-images 200

import pandas as pd
df = pd.read_csv("/tmp/manifest_smoke_test.csv")
print(f"{len(df)} rows")
df.head()

In [ ]:
# Full pull: quiet-sun (2019-2020) + active-sun (2023-2024) windows of solar
# cycle 25, so the corpus isn't skewed toward one filament-density regime.
# Adjust --day-stride based on the smoke test above and re-run this cell (it
# only builds the manifest -- nothing is downloaded until the next cell) until
# the printed row count lands in the 5,000-10,000 range.
#
# --workers controls concurrency (I/O-bound, multi-threaded) and --log-file
# streams a running "[completed/total] N frames -- elapsed/remaining" line --
# a first real run of this without threading/logging sat for the better part
# of an hour with zero visible progress, since the only signal was the final
# row count printed at completion. It also checkpoints the manifest CSV
# periodically, so a killed cell still leaves a real partial result.
!python scripts/pretrain_data/gong_halpha.py manifest \
    --sites big_bear mauna_loa \
    --date-ranges 2019-06-01:2020-06-01 2023-06-01:2024-06-01 \
    --day-stride 3 \
    --max-images 10000 \
    --workers 6 \
    --out data/raw/gong_pretrain/manifest.csv \
    --log-file data/raw/gong_pretrain/manifest_build.log

import pandas as pd
df = pd.read_csv("data/raw/gong_pretrain/manifest.csv")
print(f"{len(df)} rows -- site counts:")
print(df["site"].value_counts())

### 3. Stage 1 -- download

Resume-safe (skips files already on disk), so if the Kaggle session ends
mid-download, just re-run this cell after remounting/re-cloning -- it picks up
where it left off rather than re-downloading everything.

In [ ]:
!python scripts/pretrain_data/gong_halpha.py download \
    --manifest data/raw/gong_pretrain/manifest.csv \
    --out-dir data/raw/gong_pretrain \
    --workers 6

**Checkpoint:** once this finishes, use the Kaggle UI -- **New Dataset ->
upload `data/raw/gong_pretrain/` -> version it** as `halpha-raw` -- before
moving on, so a killed/restarted kernel doesn't lose the download.

### 4. Stage 2 -- preprocess

Deliberately minimal: converts each FITS frame to an 8-bit JPEG and nothing
else -- no limb-darkening correction, no contrast stretch beyond the per-image
min/max rescale needed to fit 16-bit data into 8-bit at all, no dedup. Output
stays at the full native 2048x2048, 1-channel resolution *and* MAGFiLO's own
JPEG format -- verified this matches MAGFiLO's own training images exactly
(same size, same single-channel format, same file format), so there's no
train/pretrain mismatch to bridge later. A ViT can't consume a 2048x2048
frame directly, though -- Stage 3's dataset samples a disk-radius-bounded crop
from these full-resolution frames per training step instead (see
`PRETRAIN_PLAN.md` section 4.1).

See `PRETRAIN_PLAN.md` section 3 -- an earlier revision of this script did
limb-darkening correction/contrast-stretch/dedup (all verified against real
data, including catching a couple of real bugs along the way); that work is
preserved in git history but not run by default anymore.

In [ ]:
!python scripts/pretrain_data/preprocess_gong.py \
    --raw-dir data/raw/gong_pretrain --out-dir data/processed/gong_pretrain \
    --processes 8 \
    --log-file data/processed/gong_pretrain/preprocess.log

### 5. Sanity-check the output before trusting it at scale

Visualize a few converted frames -- catching a wrong header key or a
corrupt download here is much cheaper than discovering it after the full
corpus is built.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

manifest = pd.read_csv("data/processed/gong_pretrain/manifest.csv")
print(f"{len(manifest)} JPEGs")

sample = manifest.sample(min(6, len(manifest)), random_state=0)
fig, axes = plt.subplots(1, len(sample), figsize=(3 * len(sample), 3))
for ax, (_, row) in zip(np.atleast_1d(axes), sample.iterrows()):
    img = np.array(Image.open(row["path"]))
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(row["site"], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

### 6. Next steps

- Use the Kaggle UI -- **New Dataset -> upload `data/processed/gong_pretrain/`
  -> version it** as `halpha-preprocessed`.
- Stage 3 (MAE pretraining) and Stage 4 (fine-tuning handoff) run in separate
  notebooks that mount these two datasets as read-only inputs -- see
  `PRETRAIN_PLAN.md` sections 4-5. This notebook's job ends here.